# 🚇 TunnelFlow-UB — public Ollama on Colab in one run

**Usage:** open this notebook → **Runtime → Run all** (`Ctrl+F9`) → wait a few minutes → copy the `http://bore.pub:<PORT>` URL from cell 1.

Run order (tunnel first, SSH only after the tunnel is verified):

| Cell | What it does |
|---|---|
| 1 | Install Bore + Ollama, start Ollama, open the tunnel, **verify it over the internet**, print the ready-to-use URL |
| 2 | (optional) SSH into Colab — **skipped automatically until the cell 1 tunnel is healthy** |
| 3 | Pull `tranhuythang9999/deepseek-v4-pro-9b-heretic:probe` and call it **through the public URL** as a real end-to-end test |
| 4 | Copy-paste snippets to call it from your machine (curl / Python / OpenAI SDK) |

### Bug fixed vs. the old `script_tunnel.sh`

```text
Real bore log line:  "2026-11-12T09:41:07.123Z  INFO bore_cli::client: listening at bore.pub:41234"
Old code:            grep -o '[0-9]\{4,5\}' | head -n 1              -> "2026"   (grabs the YEAR from the timestamp)
New code:            grep -oE 'bore\.pub:[0-9]{1,5}' | sed 's/.*://'  -> "41234"  (correct)
```

Also fixed: replaced the fixed `sleep 4` with a 30 s wait loop, guarded the log parsing so `set -e` cannot kill the
script silently, stripped ANSI colours and `\r`, and the reported port is now **verified with `curl`** through `bore.pub`
before it is printed.

> ⚠️ A free Colab runtime stops after ~12 h or when idle → just re-run the notebook to get a new port.


In [ ]:
%%bash
# ==========================================================
# CELL 1 — Ollama + Bore tunnel: one click and you are done
# ==========================================================
set -e

# ---------------- CONFIG ----------------
LOCAL_PORT=11434          # Ollama port inside Colab
REMOTE_HOST="bore.pub"    # public tunnel server
# ----------------------------------------

echo "=================================================="
echo "1/5. Installing Bore (TCP tunnel)..."
echo "=================================================="
if command -v bore > /dev/null 2>&1; then
    echo "OK: bore already installed: $(bore --version)"
else
    wget -q https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    tar -xzf bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    mv bore /usr/local/bin/
    rm -f bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    echo "OK: installed $(bore --version)"
fi

echo "=================================================="
echo "2/5. Installing dependencies..."
echo "=================================================="
apt-get update -qq
apt-get install -y -qq zstd curl ca-certificates > /dev/null

echo "=================================================="
echo "3/5. Installing Ollama..."
echo "=================================================="
if command -v ollama > /dev/null 2>&1; then
    echo "OK: ollama already installed: $(ollama --version)"
else
    # install.sh often fails at the systemctl step (Colab has no systemd) -> do not abort
    curl -fsSL https://ollama.com/install.sh | sh || echo "WARN: install.sh reported an error (usually missing systemd); checking the binary next."
fi
if ! command -v ollama > /dev/null 2>&1; then
    echo "ERROR: could not install Ollama."
    exit 1
fi

echo "=================================================="
echo "4/5. Starting Ollama in the background on 0.0.0.0:${LOCAL_PORT}..."
echo "=================================================="
# Clean up old processes, but only this port's tunnel (never kill a running SSH tunnel)
pkill -f "ollama serve" || true
pkill -f "bore local ${LOCAL_PORT} --to" || true
sleep 2

nohup env OLLAMA_HOST=0.0.0.0:${LOCAL_PORT} OLLAMA_ORIGINS="*" ollama serve > ollama.log 2>&1 &

# Wait for the API instead of a blind sleep (up to 60s)
OLLAMA_OK=0
for _ in $(seq 1 60); do
    if curl -sf "http://127.0.0.1:${LOCAL_PORT}/api/tags" > /dev/null 2>&1; then
        OLLAMA_OK=1
        break
    fi
    sleep 1
done
if [ "$OLLAMA_OK" != "1" ]; then
    echo "ERROR: Ollama is not responding. Last log lines:"
    tail -n 30 ollama.log
    exit 1
fi
echo "OK: Ollama is up at http://127.0.0.1:${LOCAL_PORT}"

echo "=================================================="
echo "5/5. Opening the Bore tunnel and reading the CORRECT public port..."
echo "=================================================="
rm -f bore.log
nohup bore local ${LOCAL_PORT} --to ${REMOTE_HOST} > bore.log 2>&1 &
BORE_PID=$!

# OLD BUG: grep -o '[0-9]\{4,5\}' on the line
#   "2026-11-12T09:41:07.123Z  INFO bore_cli::client: listening at bore.pub:41234"
#   matched the YEAR "2026" first => printed http://bore.pub:2026 (wrong port)
# FIX: only take the digits that come right after "bore.pub:"
PORT=""
for _ in $(seq 1 30); do
    PORT=$(sed 's/\x1b\[[0-9;]*m//g' bore.log 2>/dev/null \
           | tr -d '\r' \
           | grep -oE "bore\.pub:[0-9]{1,5}" \
           | sed 's/.*://' \
           | tail -n 1) || true
    if [ -n "$PORT" ]; then
        break
    fi
    sleep 1
done

if [ -z "$PORT" ]; then
    echo "WARN: could not read the port yet. Raw bore log:"
    sed 's/\x1b\[[0-9;]*m//g' bore.log | tail -n 20
    kill -0 "$BORE_PID" 2>/dev/null || echo "ERROR: the bore process already exited."
    exit 1
fi
echo "$PORT" > bore.port

# Real verification over the internet -> proves the parsed port is correct
TUNNEL_OK=0
for _ in $(seq 1 15); do
    if curl -sf --max-time 10 "http://${REMOTE_HOST}:${PORT}/api/tags" > /dev/null 2>&1; then
        TUNNEL_OK=1
        break
    fi
    sleep 2
done

echo "=================================================="
if [ "$TUNNEL_OK" = "1" ]; then
    echo "DONE: Ollama is public and reachable from outside."
    echo "API base : http://${REMOTE_HOST}:${PORT}"
    echo "Test     : curl http://${REMOTE_HOST}:${PORT}/api/tags"
    echo "Python   : base_url='http://${REMOTE_HOST}:${PORT}/v1'"
else
    echo "WARN: port found, but the public check did not pass yet (bore.pub can be slow)."
    echo "API base : http://${REMOTE_HOST}:${PORT}"
    echo "Retry in 10s: curl http://${REMOTE_HOST}:${PORT}/api/tags"
fi
echo "=================================================="


In [ ]:
%%bash
# ==========================================================
# CELL 2 (optional) — SSH into Colab.
# Runs ONLY when the Cell 1 tunnel is already healthy.
# Change ROOT_PASSWORD before using this.
# ==========================================================
ENABLE_SSH=1
ROOT_PASSWORD="${ROOT_PASSWORD:-MySecurePass123}"
export ROOT_PASSWORD

if [ "$ENABLE_SSH" != "1" ]; then
    echo "SKIP: ENABLE_SSH=0."
    exit 0
fi

# Prerequisite: the Ollama tunnel from Cell 1 must work
PORT_OLLAMA=$(cat bore.port 2>/dev/null || true)
if [ -z "$PORT_OLLAMA" ] || ! curl -sf --max-time 10 "http://bore.pub:${PORT_OLLAMA}/api/tags" > /dev/null 2>&1; then
    echo "SKIP: the Ollama tunnel is not healthy yet. Run Cell 1 first."
    exit 0
fi
echo "OK: Ollama tunnel is healthy (port ${PORT_OLLAMA}) -> setting up SSH."

echo "=================================================="
echo "1. Installing the SSH server..."
echo "=================================================="
apt-get install -y -qq openssh-server > /dev/null 2>&1 || true
echo "OK: openssh-server installed."

echo "=================================================="
echo "2. Setting the root password (non-interactive)..."
echo "=================================================="
echo "root:${ROOT_PASSWORD}" | chpasswd
echo "OK: password set."

echo "=================================================="
echo "3. Configuring sshd for a container/Colab environment..."
echo "=================================================="
mkdir -p /run/sshd
sed -i 's/#\?PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
sed -i 's/#\?PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
ssh-keygen -A > /dev/null 2>&1
grep -E "PermitRootLogin|PasswordAuthentication" /etc/ssh/sshd_config

echo "=================================================="
echo "4. Starting sshd..."
echo "=================================================="
pkill sshd || true
sleep 1
/usr/sbin/sshd
ss -tlnp 2>/dev/null | grep sshd || echo "WARN: sshd is not listening yet."

echo "=================================================="
echo "5. Opening a Bore tunnel for port 22..."
echo "=================================================="
pkill -f "bore local 22 --to" 2>/dev/null || true   # only the old SSH tunnel, keep the Ollama one
rm -f bore_ssh.log
nohup bore local 22 --to bore.pub > bore_ssh.log 2>&1 &

SSH_PORT=""
for _ in $(seq 1 30); do
    SSH_PORT=$(sed 's/\x1b\[[0-9;]*m//g' bore_ssh.log 2>/dev/null \
               | tr -d '\r' \
               | grep -oE "bore\.pub:[0-9]{1,5}" \
               | sed 's/.*://' \
               | tail -n 1) || true
    if [ -n "$SSH_PORT" ]; then
        break
    fi
    sleep 1
done

if [ -n "$SSH_PORT" ]; then
    echo "DONE: SSH public port: $SSH_PORT"
    echo "ssh -p ${SSH_PORT} root@bore.pub"
    echo "Password: ${ROOT_PASSWORD}"
else
    echo "WARN: could not read the SSH port. Raw log:"
    sed 's/\x1b\[[0-9;]*m//g' bore_ssh.log | tail -n 20
fi


In [ ]:
%%bash
# ==========================================================
# CELL 3 — Pull the model and call it THROUGH THE PUBLIC URL
# (proves the tunnel and the parsed port are really correct)
# ==========================================================
MODEL="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"

PORT=$(cat bore.port 2>/dev/null || true)
if [ -z "$PORT" ]; then
    echo "SKIP: no bore.port yet. Run Cell 1 first."
    exit 0
fi

echo "=================================================="
echo "Pulling $MODEL (the first pull takes a while)..."
echo "=================================================="
if ! ollama pull "$MODEL"; then
    echo "ERROR: pull failed for $MODEL"
    echo "Hint: if the model is private, make it public on ollama.com or push it with 'ollama push'."
    exit 1
fi

echo
echo "=================================================="
echo "Calling it through the tunnel: http://bore.pub:$PORT"
echo "=================================================="
curl -s --max-time 180 "http://bore.pub:$PORT/api/generate" \
    -d "{\"model\":\"$MODEL\",\"prompt\":\"Answer in one short sentence: what is 1+1?\",\"stream\":false}" \
    | python3 -c "import sys,json; d=json.load(sys.stdin); print('Model says:', (d.get('response') or '(empty)').strip())" \
    || echo "WARN: the test call failed, re-check the URL printed in Cell 1."

echo
echo "=================================================="
echo "Local model list:"
echo "=================================================="
ollama list


## Sample code — call it from anywhere

The port is the one printed in Cell 1 (also stored in `bore.port`). The two code cells **below** are runnable inside
Colab; the snippets after them are for your own machine — just replace `<PORT>`.

```bash
# 1) health check
curl http://bore.pub:<PORT>/api/tags

# 2) one-shot prompt (native Ollama API)
curl http://bore.pub:<PORT>/api/generate \
  -d '{"model":"tranhuythang9999/deepseek-v4-pro-9b-heretic:probe","prompt":"hello","stream":false}'
```

```python
# 3) Python - openai SDK (Ollama exposes an OpenAI-compatible API)
# pip install openai
from openai import OpenAI

client = OpenAI(base_url="http://bore.pub:<PORT>/v1", api_key="ollama")
print(client.chat.completions.create(
    model="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe",
    messages=[{"role": "user", "content": "Hello!"}],
).choices[0].message.content)
```

```python
# 4) Streaming answer
stream = client.chat.completions.create(
    model="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe",
    messages=[{"role": "user", "content": "Tell me a 3-line story."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)
```

```python
# 5) Pure requests, no SDK
import requests
r = requests.post(
    "http://bore.pub:<PORT>/api/generate",
    json={"model": "tranhuythang9999/deepseek-v4-pro-9b-heretic:probe", "prompt": "Hi", "stream": False},
    timeout=180,
)
print(r.json()["response"])
```

```bash
# 6) SSH into Colab (the SSH port is printed in Cell 2)
ssh -p <SSH_PORT> root@bore.pub
```

**Inspect or stop everything on Colab:**

```bash
pgrep -a ollama; pgrep -a bore
pkill ollama; pkill bore          # stop all tunnels and the server
```


In [ ]:
# ==========================================================
# CELL 4 — Sample client code, runs right here
# Non-stream + stream calls through the public tunnel (OpenAI SDK)
# ==========================================================
import time
from pathlib import Path

MODEL = "tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
BASE_URL = "http://bore.pub:" + Path("bore.port").read_text().strip()
print("Endpoint:", BASE_URL)
print("Model   :", MODEL)

# Ollama ignores the key, but the SDK requires one
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=BASE_URL + "/v1", api_key="ollama", timeout=180)


def ask(prompt, system="You are a concise assistant. Answer in 1-2 sentences."):
    """Single blocking request."""
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
    )
    text = (resp.choices[0].message.content or "").strip()
    print(f"[{time.time() - t0:.1f}s] {text}")
    return text


def ask_stream(prompt, system="You are a helpful assistant."):
    """Streaming request - tokens printed as they arrive."""
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
        stream=True,
    )
    print("", end="")
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
    print()


try:
    ask("What is 1+1?")
    print("\n--- streaming ---")
    ask_stream("Write 3 short lines about the ocean.")
except Exception as exc:  # noqa: BLE001
    print("Call failed:", type(exc).__name__, exc)
    print("Hint: re-run Cell 1 (the tunnel or the port may have changed).")


In [ ]:
%%bash
# ==========================================================
# CELL 5 — Same thing with plain curl (copy-paste friendly)
# ==========================================================
PORT=$(cat bore.port 2>/dev/null || true)
if [ -z "$PORT" ]; then
    echo "SKIP: no bore.port yet. Run Cell 1 first."
    exit 0
fi
MODEL="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
BASE="http://bore.pub:${PORT}"
echo "Base URL: ${BASE}"

echo
echo "1) GET /api/tags  (list models)"
curl -s --max-time 30 "${BASE}/api/tags" | head -c 400
echo

echo
echo "2) POST /api/generate  (native Ollama API)"
curl -s --max-time 180 "${BASE}/api/generate" \
    -d "{{\\"model\\":\\"${MODEL}\\",\\"prompt\\":\\"Hello in 5 words\\",\\"stream\\":false}}" \
    | python3 -c "import sys,json;d=json.load(sys.stdin);print((d.get('response') or '(empty)').strip())" \
    || echo "WARN: /api/generate failed"

echo
echo "3) POST /v1/chat/completions  (OpenAI-compatible API)"
curl -s --max-time 180 "${BASE}/v1/chat/completions" \
    -H "Content-Type: application/json" \
    -d "{{\\"model\\":\\"${MODEL}\\",\\"messages\\":[{{\\"role\\":\\"user\\",\\"content\\":\\"Say hi in 5 words\\"}}]}}" \
    | python3 -c "import sys,json;d=json.load(sys.stdin);print(d['choices'][0]['message']['content'].strip())" \
    || echo "WARN: /v1/chat/completions failed"
